# 10 · Synthetic generalisation and retarget results

Loads the tables under `runs/jepa-replenishment/synthetic/` and states the
success rules beside them. In-distribution, unseen-SKU, and
unseen-disruption are separate tables, and the holding-rate retarget is a
fourth. Those files are git-ignored. When they are
absent this notebook prints the published command that fills them and
stops. It does not plot empty numbers.

Set `JEPA_SYNTHETIC_DIR` to read a different directory. The published
log, fit, and tuning studies are in the root README section
[Latent replenishment](../README.md#latent-replenishment).


In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "pyproject.toml").exists():
    os.chdir("..")

from src.jepa.commands import EVAL_COMMAND

directory = Path(os.environ.get("JEPA_SYNTHETIC_DIR", "runs/jepa-replenishment/synthetic"))
TABLES_PRESENT = all(
    (directory / name).is_file()
    for name in (
        "in_distribution.parquet",
        "unseen_sku.parquet",
        "unseen_disruption.parquet",
        "retarget.parquet",
    )
)
if not TABLES_PRESENT:
    print(f"{directory} has no synthetic tables.")
    print("The published command that fills them is:")
    print(EVAL_COMMAND)
    refused = directory / "profit_table_refused.json"
    if refused.is_file():
        print(refused.read_text())


## In-distribution

Training SKUs, default disruption, tuner holdout, `holding_rate` 0.01.
The rule is printed with the table:
the learned arm beats the tuned anchor, and the bootstrap interval of that
paired difference excludes zero. A tie is not a pass. The table's floors are
never-order, random-cover, and least-squares-in-planner; the oracle model
in the planner is the ceiling. Order volume, service level, stockout rate,
and realised fill rate sit beside profit, with held-out rollout R² on the
same rows when the publish gate has opened. The unseen protocols below are
not pooled into this table.


In [ ]:
import pandas as pd

from src.jepa.eval import IN_DISTRIBUTION_RULE, in_distribution_passes

if TABLES_PRESENT:
    in_dist = pd.read_parquet(directory / "in_distribution.parquet")
    print(IN_DISTRIBUTION_RULE)
    print(in_dist.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
    print(
        "in_distribution_passes",
        in_distribution_passes(in_dist["latent"], in_dist["anchor"]),
    )
    warnings = directory / "comparison_warnings.json"
    if warnings.is_file():
        print(warnings.read_text())


## Unseen SKU

Evaluation SKUs only, default disruption, 32 seeds from `15_000_000`.
Same floors, oracle ceiling, and operational columns as the in-distribution
table. This is not a seed holdout of the training catalog.


In [ ]:
from src.jepa.eval import UNSEEN_SKU_SEED_OFFSET

if TABLES_PRESENT:
    unseen = pd.read_parquet(directory / "unseen_sku.parquet")
    print("protocol unseen_sku", UNSEEN_SKU_SEED_OFFSET)
    print(unseen.to_string(index=False, float_format=lambda value: f"{value:.3f}"))


## Unseen disruption

Training SKUs, the harsher disruption regime the log never samples, and
its own seed block from `16_000_000`. Same columns as the other two
protocols. The SKUs match in-distribution. The regime and the seeds do not.


In [ ]:
from src.jepa.eval import UNSEEN_DISRUPTION_SEED_OFFSET

if TABLES_PRESENT:
    shocked = pd.read_parquet(directory / "unseen_disruption.parquet")
    print("protocol unseen_disruption", UNSEEN_DISRUPTION_SEED_OFFSET)
    print(shocked.to_string(index=False, float_format=lambda value: f"{value:.3f}"))


## Retarget

Same holdout seeds, `holding_rate` 0.05, weights from the 0.01 fit. The
rule is printed with the table: the frozen planner beats the frozen
`OrderUpToPolicy` from the 0.01 study on paired `net_profit / initial_cash`.
The policy tuned at 0.05 is a ceiling and is not a pass bar.


In [ ]:
from src.jepa.eval import RETARGET_RULE, retarget_passes

if TABLES_PRESENT:
    retarget = pd.read_parquet(directory / "retarget.parquet")
    print(RETARGET_RULE)
    print(retarget.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
    print(
        "retarget_passes",
        retarget_passes(retarget["latent"], retarget["anchor"]),
    )
